In [1]:
import subprocess
from pathlib import Path

print(subprocess.run(
    ["nvidia-smi"],
    capture_output=True,
    text=True
).stdout)

print("Các Dataset trong Kaggle:")
for path in sorted(Path("/kaggle/input").iterdir()):
    print(path)

Fri Oct  2 11:47:18 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import sys

print(sys.version)

3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]


In [3]:
%pip install -q --upgrade --only-binary=:all: \
    transformers==4.44.2 \
    tokenizers==0.19.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 105.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 84.4 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 24.5 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [4]:
%pip install -q \
    omegaconf==2.3.0 \
    fairscale==0.4.4 \
    timm==0.4.12 \
    iopath==0.1.10 \
    ftfy==6.2.0 \
    wandb==0.19.11

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.4/235.4 kB 8.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 2.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 377.0/377.0 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.4/54.4 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.4/21.4 MB 79.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 7.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
urwid 4.0.3 requires wcwidth>=0.4, but you have wcwidth 0.2.14 which is incompatible.
Note: you may need to restart 

In [5]:
import timm
import iopath
import omegaconf
import fairscale
import transformers
import tokenizers
import wandb

print("Import packages: PASS")
print("transformers:", transformers.__version__)
print("tokenizers:", tokenizers.__version__)
print("wandb:", wandb.__version__)
print("timm:", timm.__version__)

Import packages: PASS
transformers: 4.44.2
tokenizers: 0.19.1
wandb: 0.19.11
timm: 0.4.12


/usr/local/lib/python3.12/dist-packages/fairscale/experimental/nn/offload.py:19: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  return torch.cuda.amp.custom_fwd(orig_func)  # type: ignore
/usr/local/lib/python3.12/dist-packages/fairscale/experimental/nn/offload.py:30: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  return torch.cuda.amp.custom_bwd(orig_func)  # type: ignore


In [6]:
import torch

print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

CUDA: True
GPU: Tesla T4


In [8]:
from pathlib import Path
import os

INPUT_ROOT = Path("/kaggle/input")

code_paths = []
fashioniq_paths = []

for current_root, dir_names, file_names in os.walk(INPUT_ROOT):
    current = Path(current_root)

    # Kiểm tra FashionIQ trước khi loại thư mục images
    required_dirs = {"captions", "images", "image_splits"}

    if required_dirs.issubset(set(dir_names)):
        fashioniq_paths.append(current)

    # Kiểm tra Code QuRe-LoRA
    if (
        "train_qure.py" in file_names
        and (current / "configs").is_dir()
        and (current / "models").is_dir()
    ):
        code_paths.append(current)

    # Không đi sâu vào thư mục ảnh
    dir_names[:] = [
        name for name in dir_names
        if name not in {
            "images",
            "image_data",
            "__pycache__",
            ".git"
        }
    ]

print("Các CODE_ROOT tìm được:")
for path in sorted(set(code_paths)):
    print(path)

print("\nCác FASHIONIQ_ROOT tìm được:")
for path in sorted(set(fashioniq_paths)):
    print(path)

print("\nSố CODE_ROOT:", len(set(code_paths)))
print("Số FASHIONIQ_ROOT:", len(set(fashioniq_paths)))

Các CODE_ROOT tìm được:
/kaggle/input/datasets/handq2212/qure-lora-code/Code

Các FASHIONIQ_ROOT tìm được:
/kaggle/input/datasets/handq2212/fashingiq/fashing_iq

Số CODE_ROOT: 1
Số FASHIONIQ_ROOT: 1


In [9]:
from pathlib import Path

CODE_ROOT = Path(
    "/kaggle/input/datasets/handq2212/qure-lora-code/Code"
)

FASHIONIQ_ROOT = Path(
    "/kaggle/input/datasets/handq2212/fashingiq/fashing_iq"
)

assert (CODE_ROOT / "train_qure.py").is_file(), CODE_ROOT
assert (CODE_ROOT / "configs").is_dir(), CODE_ROOT

assert (FASHIONIQ_ROOT / "captions").is_dir(), FASHIONIQ_ROOT
assert (FASHIONIQ_ROOT / "images").is_dir(), FASHIONIQ_ROOT
assert (FASHIONIQ_ROOT / "image_splits").is_dir(), FASHIONIQ_ROOT

print("CODE_ROOT:", CODE_ROOT)
print("FASHIONIQ_ROOT:", FASHIONIQ_ROOT)
print("Path configuration: PASS")

CODE_ROOT: /kaggle/input/datasets/handq2212/qure-lora-code/Code
FASHIONIQ_ROOT: /kaggle/input/datasets/handq2212/fashingiq/fashing_iq
Path configuration: PASS


In [10]:
print(
    "Images:",
    len(list((FASHIONIQ_ROOT / "images").glob("*.png")))
)

print(
    "Captions:",
    len(list((FASHIONIQ_ROOT / "captions").glob("*.json")))
)

print(
    "Image splits:",
    len(list((FASHIONIQ_ROOT / "image_splits").glob("*.json")))
)

Images: 74381
Captions: 9
Image splits: 9


In [11]:
bert_snapshot_root = (
    CODE_ROOT
    / "checkpoints"
    / "hf_cache"
    / "hub"
    / "models--bert-base-uncased"
    / "snapshots"
)

bert_snapshots = sorted(
    path for path in bert_snapshot_root.iterdir()
    if path.is_dir()
)

assert bert_snapshots, "Không tìm thấy BERT snapshot"

BERT_PATH = bert_snapshots[-1]

assert (BERT_PATH / "config.json").is_file()
assert (BERT_PATH / "vocab.txt").is_file()

print("BERT_PATH:", BERT_PATH)

BERT_PATH: /kaggle/input/datasets/handq2212/qure-lora-code/Code/checkpoints/hf_cache/hub/models--bert-base-uncased/snapshots/86b5e0934494bd15c9632b12f734a8a67f723594


In [12]:
import os
import sys
import json
import torch

os.environ["QURe_CHECKPOINT_ROOT"] = str(
    CODE_ROOT / "checkpoints"
)

os.environ["BLIP2_BERT_PATH"] = str(BERT_PATH)

sys.path.insert(0, str(CODE_ROOT))

with open(
    CODE_ROOT / "configs" / "fashionIQ" / "train.json",
    "r",
    encoding="utf-8",
) as f:
    config = json.load(f)

config["fashioniq_root"] = str(FASHIONIQ_ROOT)
config["experiment_dir"] = "/kaggle/working/qure_lora_smoke"
config["device_idx"] = "0"
config["use_lora"] = True
config["use_wandb"] = False
config["batch_size"] = 8

device = torch.device("cuda:0")

from models import create_qure_models

model, txt_processors = create_qure_models(
    config,
    device,
)

print("Model loaded successfully")
print(
    "LoRA configured:",
    getattr(model, "_lora_configured", False),
)

trainable = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

total = sum(
    p.numel()
    for p in model.parameters()
)

print("Trainable parameters:", trainable)
print("Total parameters:", total)

blip2 pretrain_vitL training


/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


freeze vision encoder


Some weights of BertLMHeadModel were not initialized from the model checkpoint at /kaggle/input/datasets/handq2212/qure-lora-code/Code/checkpoints/hf_cache/hub/models--bert-base-uncased/snapshots/86b5e0934494bd15c9632b12f734a8a67f723594 and are newly initialized: ['bert.encoder.layer.0.crossattention.output.LayerNorm.bias', 'bert.encoder.layer.0.crossattention.output.LayerNorm.weight', 'bert.encoder.layer.0.crossattention.output.dense.bias', 'bert.encoder.layer.0.crossattention.output.dense.weight', 'bert.encoder.layer.0.crossattention.self.key.bias', 'bert.encoder.layer.0.crossattention.self.key.weight', 'bert.encoder.layer.0.crossattention.self.query.bias', 'bert.encoder.layer.0.crossattention.self.query.weight', 'bert.encoder.layer.0.crossattention.self.value.bias', 'bert.encoder.layer.0.crossattention.self.value.weight', 'bert.encoder.layer.0.intermediate_query.dense.bias', 'bert.encoder.layer.0.intermediate_query.dense.weight', 'bert.encoder.layer.0.output_query.LayerNorm.bias', '

LoRA enabled: trainable=1,329,665, total=474,656,574
Model loaded successfully
LoRA configured: True
Trainable parameters: 1329665
Total parameters: 474656574


In [13]:
import os
import gc
import json
import torch
from pathlib import Path

from lora.multi_adapter import MultiAdapterLinear

# Kiểm tra cấu trúc LoRA
assert model._lora_configured
assert len(model._lora_target_names) == 36
assert model.query_tokens.requires_grad

lora_layers = [
    module
    for module in model.Qformer.bert.modules()
    if isinstance(module, MultiAdapterLinear)
]

assert len(lora_layers) == 36

assert all(
    not parameter.requires_grad
    for parameter in model.visual_encoder.parameters()
)

assert all(
    parameter.requires_grad
    for parameter in model.text_proj.parameters()
)

assert all(
    parameter.requires_grad
    for parameter in model.vision_proj.parameters()
)

print("LoRA structural check: PASS")
print("LoRA target layers:", len(lora_layers))
print("Trainable parameters:", sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
))

# Tạo config training thật
source_config = CODE_ROOT / "configs" / "fashionIQ" / "train.json"
run_config = Path("/kaggle/working/qure_lora_fashioniq.json")

with open(source_config, "r", encoding="utf-8") as file:
    train_config = json.load(file)

train_config.update({
    "fashioniq_root": str(FASHIONIQ_ROOT),
    "experiment_dir": "/kaggle/working/qure_lora_runs",
    "epoch": 30,
    "batch_size": 8,
    "num_workers": 0,
    "device_idx": "0",
    "use_lora": True,
    "use_wandb": False,
    "mode": "train",
    "dataset": "fashionIQ",
})

# Không resume nhầm checkpoint cũ
train_config.pop("resume_path", None)
train_config.pop("checkpoint_path", None)

with open(run_config, "w", encoding="utf-8") as file:
    json.dump(train_config, file, indent=2)

# Tắt hoàn toàn W&B trong subprocess training
os.environ["WANDB_MODE"] = "disabled"

print("\nTraining config:")
print("epoch:", train_config["epoch"])
print("batch_size:", train_config["batch_size"])
print("num_workers:", train_config["num_workers"])
print("use_lora:", train_config["use_lora"])
print("use_wandb:", train_config["use_wandb"])
print("fashioniq_root:", train_config["fashioniq_root"])
print("config path:", run_config)

# Giải phóng model smoke test trước khi mở process training
del model
del txt_processors
gc.collect()
torch.cuda.empty_cache()

print("\nGPU memory released")
print("Preparation for training: PASS")

LoRA structural check: PASS
LoRA target layers: 36
Trainable parameters: 1329665

Training config:
epoch: 30
batch_size: 8
num_workers: 0
use_lora: True
use_wandb: False
fashioniq_root: /kaggle/input/datasets/handq2212/fashingiq/fashing_iq
config path: /kaggle/working/qure_lora_fashioniq.json

GPU memory released
Preparation for training: PASS


In [16]:
from pathlib import Path
import os
import re
import shutil

PATCHED_CODE_ROOT = Path(
    "/kaggle/working/qure_lora_code_fresh"
)

shutil.copytree(
    CODE_ROOT,
    PATCHED_CODE_ROOT,
    dirs_exist_ok=True,
    ignore=shutil.ignore_patterns(
        "checkpoints",
        "__pycache__",
    ),
)

# Bổ sung CLIP checkpoint vì clip_vit.py tìm tại repository/checkpoints
src_clip = CODE_ROOT / "checkpoints" / "clip_vit_L.pth"
dst_dir = PATCHED_CODE_ROOT / "checkpoints"
dst_clip = dst_dir / "clip_vit_L.pth"

dst_dir.mkdir(parents=True, exist_ok=True)

if not dst_clip.exists():
    try:
        os.symlink(src_clip, dst_clip)
    except OSError:
        shutil.copy2(src_clip, dst_clip)

assert dst_clip.is_file()

# Patch checkpoint RNG cho resume về sau
checkpoint_file = PATCHED_CODE_ROOT / "lora" / "checkpoint.py"
checkpoint_source = checkpoint_file.read_text(encoding="utf-8")

checkpoint_source = checkpoint_source.replace(
    'torch.set_rng_state(state["torch"])',
    'torch.set_rng_state(state["torch"].detach().cpu())',
)

checkpoint_source = checkpoint_source.replace(
    'torch.cuda.set_rng_state_all(state["cuda"])',
    'torch.cuda.set_rng_state_all(['
    'item.detach().cpu() for item in state["cuda"]'
    '])',
)

checkpoint_file.write_text(
    checkpoint_source,
    encoding="utf-8",
)

# Patch FashionIQ
fashioniq_file = (
    PATCHED_CODE_ROOT
    / "data"
    / "fashionIQ.py"
)

source = fashioniq_file.read_text(encoding="utf-8")

# Giảm bộ nhớ DataLoader rerank
source, count = re.subn(
    r"DataLoader\((sample_dataset|query_dataset), "
    r"batch_size=\d+, shuffle=False, num_workers=\d+, "
    r"pin_memory=True, collate_fn=collate_fn\)",
    r"DataLoader(\1, batch_size=8, shuffle=False, "
    r"num_workers=0, pin_memory=False, collate_fn=collate_fn)",
    source,
)

assert count == 2, f"Đã patch {count} rerank loader, cần đúng 2"

# Tránh tạo toàn bộ ma trận query x gallery x 32
chunk_helper = r'''
class _ChunkedScoreAccessor:
    def __init__(self, model, query_features, target_features, chunk_size=8):
        self.model = model
        self.query_features = query_features
        self.target_features = target_features
        self.chunk_size = max(1, int(chunk_size))
        self.cached_start = None
        self.cached_scores = None

    def __getitem__(self, index):
        index = int(index)
        chunk_start = (index // self.chunk_size) * self.chunk_size

        if self.cached_start != chunk_start:
            chunk_end = min(
                chunk_start + self.chunk_size,
                self.query_features.shape[0],
            )

            self.cached_scores = self.model.score(
                self.query_features[chunk_start:chunk_end],
                self.target_features,
            )
            self.cached_start = chunk_start

        return self.cached_scores[index - chunk_start]
'''

if "_ChunkedScoreAccessor" not in source:
    marker = "class FashionIQDataset(Dataset):"
    assert marker in source

    source = source.replace(
        marker,
        chunk_helper + "\n\n" + marker,
        1,
    )

source = source.replace(
    "        scores = model.score(predicted_features, index_features)",
    """        scores = _ChunkedScoreAccessor(
            model,
            predicted_features,
            index_features,
            chunk_size=int(configs.get("rerank_score_chunk_size", 8)),
        )""",
    1,
)

# Không gọi wandb.log khi use_wandb=False
source = source.replace(
    'if configs["experiment_description"] != \'debug\':',
    'if configs.get("use_wandb", False):',
)

source = source.replace(
    'if configs["experiment_description"] != "debug":',
    'if configs.get("use_wandb", False):',
)

fashioniq_file.write_text(
    source,
    encoding="utf-8",
)

print("Fresh patched Code: PASS")
print(PATCHED_CODE_ROOT)

Fresh patched Code: PASS
/kaggle/working/qure_lora_code_fresh


In [ ]:
from pathlib import Path
import json

assert PATCHED_CODE_ROOT.is_dir()
assert run_config.is_file()

# Đảm bảo evaluation chạy sau mỗi epoch
with open(run_config, "r", encoding="utf-8") as file:
    training_config = json.load(file)

training_config["eval_frequency"] = 1

with open(run_config, "w", encoding="utf-8") as file:
    json.dump(training_config, file, indent=2)

assert training_config["eval_frequency"] == 1

# --------------------------------------------------
# Patch eval_utils.py để in Recall sau mỗi epoch
# --------------------------------------------------

eval_file = (
    PATCHED_CODE_ROOT
    / "evaluate_qure"
    / "eval_utils.py"
)

assert eval_file.is_file()

eval_source = eval_file.read_text(encoding="utf-8")

if "import json" not in eval_source:
    eval_source = eval_source.replace(
        "from statistics import mean",
        "from statistics import mean\nimport json",
        1,
    )

if "[VALIDATION EPOCH" not in eval_source:
    old_eval_return = (
        "    return results_dict\n\n\ndef eval_cirr"
    )

    new_eval_return = """    print(f"\\n[VALIDATION EPOCH {epoch}]")
    print(json.dumps(results_dict, indent=2))

    return results_dict


def eval_cirr"""

    assert old_eval_return in eval_source

    eval_source = eval_source.replace(
        old_eval_return,
        new_eval_return,
        1,
    )

eval_file.write_text(
    eval_source,
    encoding="utf-8",
)

# --------------------------------------------------
# Patch train_qure.py để đánh dấu thời điểm validation
# --------------------------------------------------

train_file = PATCHED_CODE_ROOT / "train_qure.py"

assert train_file.is_file()

train_source = train_file.read_text(encoding="utf-8")

if "[VALIDATION]" not in train_source:
    old_eval_condition = (
        "        if (epoch + 1) % eval_frequency == 0:\n"
    )

    new_eval_condition = """        if (epoch + 1) % eval_frequency == 0:
            print(
                f"\\n[VALIDATION] epoch "
                f"{epoch + 1}/{max_epochs}"
            )
"""

    assert old_eval_condition in train_source

    train_source = train_source.replace(
        old_eval_condition,
        new_eval_condition,
        1,
    )

train_file.write_text(
    train_source,
    encoding="utf-8",
)

# Kiểm tra patch
assert "[VALIDATION EPOCH" in eval_source
assert "[VALIDATION]" in train_source
assert training_config["batch_size"] == 8

print("Validation logging patch: PASS")
print("eval_frequency:", training_config["eval_frequency"])
print("batch_size:", training_config["batch_size"])
print("Code:", PATCHED_CODE_ROOT)
print("Config:", run_config)

In [ ]:
import os
import sys
import json
import shutil
import zipfile
import subprocess
from pathlib import Path

assert PATCHED_CODE_ROOT.is_dir()
assert run_config.is_file()

INPUT_ROOT = Path("/kaggle/input")

# Tìm file checkpoint nếu có
file_candidates = sorted(
    list(INPUT_ROOT.rglob("epoch_019.pt"))
    + list(INPUT_ROOT.rglob("epoch_019.zip"))
)

# Tìm checkpoint đang ở dạng thư mục
dir_candidates = sorted(
    path
    for path in INPUT_ROOT.rglob("epoch_019")
    if path.is_dir() and (path / "data.pkl").is_file()
)

resume_checkpoint = Path("/kaggle/working/epoch_019.pt")

if file_candidates:
    source_checkpoint = file_candidates[0]
    shutil.copy2(source_checkpoint, resume_checkpoint)
    print("Found checkpoint file:", source_checkpoint)

elif dir_candidates:
    checkpoint_dir = dir_candidates[0]
    print("Found extracted checkpoint directory:", checkpoint_dir)

    # Đóng gói lại đúng cấu trúc PyTorch checkpoint
    with zipfile.ZipFile(
        resume_checkpoint,
        "w",
        compression=zipfile.ZIP_STORED,
    ) as archive:
        for file_path in checkpoint_dir.rglob("*"):
            if file_path.is_file():
                relative_path = file_path.relative_to(checkpoint_dir)
                archive_name = (
                    Path(checkpoint_dir.name) / relative_path
                ).as_posix()
                archive.write(file_path, archive_name)

    print("Checkpoint repacked:", resume_checkpoint)

else:
    raise FileNotFoundError(
        "Không tìm thấy epoch_019.pt, epoch_019.zip "
        "hoặc thư mục epoch_019 trong /kaggle/input"
    )

assert resume_checkpoint.is_file()
assert zipfile.is_zipfile(resume_checkpoint)

# Cập nhật config resume
with open(run_config, "r", encoding="utf-8") as file:
    resume_config = json.load(file)

resume_config.update({
    "epoch": 30,
    "batch_size": 8,
    "num_workers": 0,
    "eval_frequency": 1,
    "resume_path": str(resume_checkpoint),
})

resume_config.pop("checkpoint_path", None)

with open(run_config, "w", encoding="utf-8") as file:
    json.dump(resume_config, file, indent=2)

# Tìm BERT snapshot
bert_snapshot_root = (
    CODE_ROOT
    / "checkpoints"
    / "hf_cache"
    / "hub"
    / "models--bert-base-uncased"
    / "snapshots"
)

bert_snapshots = sorted(
    path for path in bert_snapshot_root.iterdir()
    if path.is_dir()
)

assert bert_snapshots
BERT_PATH = bert_snapshots[-1]

# Môi trường train
train_env = os.environ.copy()
train_env["QURe_CHECKPOINT_ROOT"] = str(
    CODE_ROOT / "checkpoints"
)
train_env["BLIP2_BERT_PATH"] = str(BERT_PATH)
train_env["WANDB_MODE"] = "disabled"
train_env["PYTHONUNBUFFERED"] = "1"
train_env["PYTHONPATH"] = str(PATCHED_CODE_ROOT)

train_command = [
    sys.executable,
    "-u",
    str(PATCHED_CODE_ROOT / "train_qure.py"),
    "--config_path",
    str(run_config),
]

print("Resume checkpoint:", resume_checkpoint)
print("Expected next epoch: 20")
print("Validation: every epoch")
print("Batch size:", resume_config["batch_size"])

result = subprocess.run(
    train_command,
    cwd=str(PATCHED_CODE_ROOT),
    env=train_env,
)

if result.returncode != 0:
    raise RuntimeError(
        f"Resume training failed with exit code {result.returncode}"
    )

print("Resume training completed successfully")